# Capstone (jumeau C#) — ce que la sous-série PostTraining démontre

**Navigation** : [Version Python](RL-20-Capstone-PostTraining-Python.ipynb) · [README de la série](README.md)

Cette sous-série a produit onze carnets. Chacun a entraîné quelque chose, mesuré quelque chose, et committé ses sorties. Le présent carnet ne réentraîne rien : il **lit les sorties committées**, en extrait les nombres, et calcule ce qu'ils autorisent à affirmer — et ce qu'ils n'autorisent pas.

C'est le geste que la sous-série ne pouvait pas faire sur elle-même, parce qu'il demande de tenir les onze protocoles côte à côte, et une chose désagréable : comparer chaque écart annoncé à la résolution de l'instrument qui l'a mesuré.

Il en sort une lecture unique en son genre dans la série : **aucune des quatre comparaisons de la sous-série ne survit à ses propres barres d'erreur**, et la raison est chaque fois la même — l'instrument mesure à une résolution plus grossière que l'écart qu'on lui demande de départager. Ce n'est pas un défaut des carnets : c'est la propriété des petits mondes qu'ils instruisent, et c'est précisément ce qu'un capstone doit rendre lisible.

Exécution CPU, sans GPU ni checkpoint : la matière première est déjà dans les fichiers committés.

**Navigation** : [README série](README.md) — précédent : [RL-19-Auto-Jeu-AlphaZero](RL-19-Auto-Jeu-AlphaZero-Python.ipynb) — sous-série : [PostTraining](PostTraining/README.md)

## Ce que ce notebook suppose

Onze carnets, tous exécutés et committés avec leurs sorties. L'analyse en dépend entièrement : elle ne recalcule aucun entraînement, elle relit des preuves.

| Carnet | Ce qu'il établit | Noyau d'exécution |
|---|---|---|
| [RLPT-00](PostTraining/RLPT-00-Reward-Model-Bradley-Terry-Python.ipynb) | reward model Bradley-Terry écrit à la main, plafond de Bayes empirique, calibration | python3 (CPU) |
| [RLPT-00b](PostTraining/RLPT-00b-Biais-Dataset-Preferences-Python.ipynb) | biais de longueur dans un dataset de préférences, contrôle de longueur | python3 (CPU) |
| [RLPT-00c](PostTraining/RLPT-00c-Anatomie-Reward-Hacking-Python.ipynb) | anatomie du reward hacking, seuil analytique du rapport longueur/correction | coursia-ml-training (GPU) |
| [RLPT-00d](PostTraining/RLPT-00d-Reward-Trainer-TRL-Python.ipynb) | `RewardTrainer` de TRL contre la boucle maison, à budget égal | python3 (CPU) |
| [RLPT-00e](PostTraining/RLPT-00e-DPO-TRL-Python.ipynb) | DPO maison contre DPO TRL, même monde, même graines | python3 (CPU) |
| [RLPT-00f](PostTraining/RLPT-00f-GRPO-TRL-contre-PPO-Maison-Python.ipynb) | GRPO de TRL contre PPO maison, quatre graines appariées | python3 (CPU) |
| [RLPT-00g](PostTraining/RLPT-00g-PPO-TRL-Experimental-Python.ipynb) | PPO expérimental de TRL, fenêtre de versions et bras négatif | python3 (CPU) |
| [RLPT-01](PostTraining/RLPT-01-PPO-RLHF-Petit-Modele-Python.ipynb) | PPO/RLHF sur petit modèle, quatre graines, balayage de `beta` | python3 (CPU) |
| [RLPT-02](PostTraining/RLPT-02-GRPO-Minimal-Python.ipynb) | GRPO minimal sur LoRA, quatre graines | coursia-ml-training (GPU) |
| [RLPT-03](PostTraining/RLPT-03-Reward-Hacking-Python.ipynb) | contamination du few-shot et hacking mesuré sur un adaptateur réel | coursia-ml-training (GPU) |
| [RLPT-04](PostTraining/RLPT-04-DPO-Offline-contre-GRPO-Online-Python.ipynb) | DPO hors ligne contre GRPO en ligne, même budget, même graine | coursia-ml-training (GPU) |

Les quatre carnets GPU ne sont **pas** réexécutés ici : leur entraînement a déjà eu lieu, et le refaire produirait un autre run, pas une meilleure preuve. Ce que ce carnet ajoute est d'un autre ordre — l'arithmétique qui manque entre les lignes.

## 1. L'instrument : lire les sorties committées

Tout commence par une décision de méthode. Les nombres utilisés plus bas ne sont **pas recopiés** : ils sont **extraits** des sorties committées, par recherche de libellé. Si une sortie change, l'analyse suit — et si un libellé disparaît, l'extraction rend `None` au lieu de rendre un nombre inventé.

L'extraction est insensible aux accents, parce que les sorties du dépôt mélangent les deux orthographes selon le carnet.

In [1]:
// Instrument du capstone : tout ce qui suit lit les sorties committées des
// onze carnets de la sous-série -- rien ici n'entraîne ni ne recalcule un modèle.
using System;
using System.Collections.Generic;
using System.Globalization;
using System.IO;
using System.Linq;
using System.Text;
using System.Text.Json;
using System.Text.RegularExpressions;

// Le carnet peut être exécuté depuis la racine du dépôt ou depuis la série :
// on cherche l'ancêtre qui porte MyIA.AI.Notebooks.
static DirectoryInfo TrouverRacine()
{
    var courant = new DirectoryInfo(Directory.GetCurrentDirectory());
    for (var c = courant; c != null; c = c.Parent)
    {
        if (Directory.Exists(Path.Combine(c.FullName, "MyIA.AI.Notebooks")))
        {
            return new DirectoryInfo(Path.Combine(c.FullName, "MyIA.AI.Notebooks", "RL"));
        }
    }
    return new DirectoryInfo(Path.GetFullPath("MyIA.AI.Notebooks/RL"));
}

var RACINE = TrouverRacine();
var SOUS_SERIE = new DirectoryInfo(Path.Combine(RACINE.FullName, "PostTraining"));

// Les onze carnets de la sous-série, dans l'ordre de lecture du README.
var CARNETS = new[]
{
    "RLPT-00-Reward-Model-Bradley-Terry-Python.ipynb",
    "RLPT-00b-Biais-Dataset-Preferences-Python.ipynb",
    "RLPT-00c-Anatomie-Reward-Hacking-Python.ipynb",
    "RLPT-00d-Reward-Trainer-TRL-Python.ipynb",
    "RLPT-00e-DPO-TRL-Python.ipynb",
    "RLPT-00f-GRPO-TRL-contre-PPO-Maison-Python.ipynb",
    "RLPT-00g-PPO-TRL-Experimental-Python.ipynb",
    "RLPT-01-PPO-RLHF-Petit-Modele-Python.ipynb",
    "RLPT-02-GRPO-Minimal-Python.ipynb",
    "RLPT-03-Reward-Hacking-Python.ipynb",
    "RLPT-04-DPO-Offline-contre-GRPO-Online-Python.ipynb",
};

static JsonElement Charger(string nom)
{
    var chemin = Path.Combine(TrouverRacine().FullName, "PostTraining", nom);
    var texte = File.ReadAllText(chemin, Encoding.UTF8);
    using var doc = JsonDocument.Parse(texte);
    return doc.RootElement.Clone();
}

static string SortieTexte(JsonElement carnet)
{
    // Concatène les sorties texte de toutes les cellules de code.
    var morceaux = new List<string>();
    foreach (var cellule in carnet.GetProperty("cells").EnumerateArray())
    {
        if (!cellule.TryGetProperty("cell_type", out var ct) || ct.GetString() != "code")
        {
            continue;
        }
        if (!cellule.TryGetProperty("outputs", out var sorties))
        {
            continue;
        }
        foreach (var sortie in sorties.EnumerateArray())
        {
            if (!sortie.TryGetProperty("text", out var texte))
            {
                continue;
            }
            if (texte.ValueKind == JsonValueKind.Array)
            {
                morceaux.Add(string.Concat(texte.EnumerateArray().Select(t => t.GetString())));
            }
            else if (texte.ValueKind == JsonValueKind.String)
            {
                morceaux.Add(texte.GetString() ?? "");
            }
        }
    }
    return string.Join("\n", morceaux);
}

static string SansAccent(string texte)
{
    // Les sorties du dépôt mélangent accents avec et sans : la recherche de
    // libellé doit indifférencer les deux orthographes.
    var decompose = texte.Normalize(NormalizationForm.FormD);
    var sb = new StringBuilder(decompose.Length);
    foreach (var c in decompose)
    {
        if (CharUnicodeInfo.GetUnicodeCategory(c) != UnicodeCategory.NonSpacingMark)
        {
            sb.Append(c);
        }
    }
    return sb.ToString();
}

static readonly Regex NOMBRE = new Regex(@"[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?", RegexOptions.Compiled);

static double?[] NombresApres(string texte, string libelle, int combien = 1)
{
    // Les `combien` premiers nombres qui suivent `libelle` (null si absent).
    var plat = SansAccent(texte);
    var libellePlat = SansAccent(libelle);
    var ancre = plat.IndexOf(libellePlat, StringComparison.Ordinal);
    if (ancre < 0)
    {
        return Enumerable.Repeat<double?>(null, combien).ToArray();
    }
    // On extrait depuis la fin du libellé : le nombre cherché est la valeur
    // ANNONCÉE par la ligne, pas un chiffre du libellé lui-même.
    var debut = ancre + libellePlat.Length;
    var trouves = NOMBRE.Matches(plat, debut)
        .Cast<Match>()
        .Select(m => (double?)double.Parse(m.Value, CultureInfo.InvariantCulture))
        .ToList();
    while (trouves.Count < combien) trouves.Add(null);
    return trouves.Take(combien).ToArray();
}

static double? NombreApres(string texte, string libelle) => NombresApres(texte, libelle, 1)[0];

// Le corpus : le texte de toutes les sorties, par carnet.
var TXT = new Dictionary<string, string>();
foreach (var nom in CARNETS)
{
    TXT[nom] = SortieTexte(Charger(nom));
}
Console.WriteLine($"{TXT.Count} carnets charges | {TXT.Values.Sum(v => v.Length)} caracteres de sortie texte");


11 carnets charges | 59613 caracteres de sortie texte


In [2]:
// Preuve de charge avant toute extraction : chaque carnet source doit porter
// son execution_count et ses sorties, sinon la matière première du capstone est creuse.
var lignes = new List<(string Court, string Noyau, int Code, int Executees, int Sorties)>();
int non_executees = 0;
foreach (var nom in CARNETS)
{
    var carnet = Charger(nom);
    var code = carnet.GetProperty("cells").EnumerateArray()
        .Where(c => c.TryGetProperty("cell_type", out var ct) && ct.GetString() == "code")
        .ToList();
    int executees = code.Count(c =>
        c.TryGetProperty("execution_count", out var ec) && ec.ValueKind != JsonValueKind.Null);
    int sorties = code.Sum(c =>
        c.TryGetProperty("outputs", out var o) && o.ValueKind == JsonValueKind.Array
            ? o.GetArrayLength()
            : 0);
    string noyau = carnet.TryGetProperty("metadata", out var md)
        && md.TryGetProperty("kernelspec", out var ks)
        && ks.TryGetProperty("name", out var kn)
            ? kn.GetString() ?? "?"
            : "?";
    // Court = deux premiers segments du nom, séparés par un tiret.
    var segments = nom.Split('-');
    var court = string.Join("-", segments.Take(2));
    non_executees += code.Count - executees;
    lignes.Add((court, noyau, code.Count, executees, sorties));
}

Console.WriteLine($"{"carnet",-9} {"noyau",-21} {"code",5} {"exec",5} {"sorties",8}");
Console.WriteLine(new string('-', 52));
foreach (var (court, noyau, total, executees, sorties) in lignes)
{
    Console.WriteLine($"{court,-9} {noyau,-21} {total,5} {executees,5} {sorties,8}");
}
Console.WriteLine();
Console.WriteLine($"cellules de code sans execution_count : {non_executees}");
// Garde d'intégrité : la suite du carnet ne lit que des sorties réelles.
if (non_executees != 0)
{
    throw new InvalidOperationException(
        "une sortie committée sans execution_count : la preuve est incomplète");
}
Console.WriteLine("integrite OK : chaque cellule de code porte son execution_count et ses sorties.");


carnet    noyau                  code  exec  sorties


----------------------------------------------------


RLPT-00   python3                  15    15       22


RLPT-00b  python3                  13    13       14


RLPT-00c  coursia-ml-training      21    21       37


RLPT-00d  python3                  15    15       26


... (5 outputs left out)

RLPT-03   coursia-ml-training      12    12       38


RLPT-04   coursia-ml-training      12    12       62


cellules de code sans execution_count : 0


integrite OK : chaque cellule de code porte son execution_count et ses sorties.


### Lecture du tableau

Les onze carnets portent **zéro** cellule de code non exécutée. Ce n'est pas une formalité : c'est ce qui autorise tout ce qui suit. Une sortie committée sans `execution_count` serait une capture d'écran, pas une mesure — et l'analyse qui s'appuie dessus n'aurait aucune valeur de preuve.

Le tableau dit aussi autre chose, qui compte pour la suite : **sept carnets tournent sur CPU, quatre sur le noyau GPU `coursia-ml-training`**. Les comparaisons les plus coûteuses en calcul sont donc celles des carnets GPU, et ce sont elles qui disposent des plus petits jeux de test — nous y revenons en section 2.

## 2. Quatre comparaisons, et la résolution de leur instrument

La sous-série annonce plusieurs écarts. On les reprend un par un, avec la seule question qui tranche : **l'écart est-il plus grand que ce que l'instrument peut résoudre ?**

Quatre instruments différents sont à l'œuvre : des barres d'erreur inter-graines, un protocole apparié, un jeu de test de 48 items, et une exactitude qui ne bouge plus du tout.

In [3]:
static double EcartType(IReadOnlyList<double> valeurs)
{
    // Écart-type d'échantillon (dénominateur n-1) : la convention des carnets
    // sources quand ils rapportent "+- sd" sur peu de graines.
    int n = valeurs.Count;
    double moyenne = valeurs.Average();
    double somme = valeurs.Sum(v => (v - moyenne) * (v - moyenne));
    return Math.Sqrt(somme / (n - 1));
}

static string Inv(double v, string format) => v.ToString(format, CultureInfo.InvariantCulture);

// --- Comparaison 1 : le dénominateur du z (une barre ou les deux) ---
Console.WriteLine("=== Comparaison 1 : RLPT-00d, RewardTrainer TRL contre la reference A.1 ===");
var txt1 = TXT["RLPT-00d-Reward-Trainer-TRL-Python.ipynb"];
var apparies = Regex.Matches(
    SansAccent(txt1),
    @"seed\s+\d+\s*:\s*acc\s+([0-9.]+)\s*\(Bayes\s+([0-9.]+)\)");
var acc_trl = apparies.Cast<Match>()
    .Select(m => double.Parse(m.Groups[1].Value, CultureInfo.InvariantCulture)).ToList();
var refPaire = NombresApres(txt1, "reference A.1", 2);
double acc_ref = refPaire[0]!.Value;
double sd_ref = refPaire[1]!.Value;
double moy_trl = acc_trl.Average();
double sd_trl = EcartType(acc_trl);
// Deux dénominateurs pour le même écart : la barre A.1 seule (chiffre annoncé),
// puis les deux bras combinés (l'instrument honnête).
double z_deux_barres = (moy_trl - acc_ref) / Math.Sqrt(sd_trl * sd_trl + sd_ref * sd_ref);
double z_une_barre = (moy_trl - acc_ref) / sd_ref;
Console.WriteLine($"  TRL           : {Inv(moy_trl, "0.0000")} +- {Inv(sd_trl, "0.0000")}  (n={acc_trl.Count} graines)");
Console.WriteLine($"  reference A.1 : {Inv(acc_ref, "0.0000")} +- {Inv(sd_ref, "0.0000")}");
Console.WriteLine($"  ecart         : {Inv(moy_trl - acc_ref, "+0.0000;-0.0000")}");
Console.WriteLine($"  z sur la seule barre A.1 : {Inv(z_une_barre, "+0.00;-0.00")}   <- le chiffre annonce par le carnet");
Console.WriteLine($"  z en combinant LES DEUX  : {Inv(z_deux_barres, "+0.00;-0.00")}   <- le chiffre honnete");
Console.WriteLine();

// --- Comparaison 2 : graines appariées -> test apparié ---
Console.WriteLine("=== Comparaison 2 : RLPT-00f, GRPO TRL contre PPO maison, graines appariees ===");
var txt2 = TXT["RLPT-00f-GRPO-TRL-contre-PPO-Maison-Python.ipynb"];
// Chaque seed a tourné sur les deux bras : la différence par graine est la
// bonne unité statistique, pas les deux écarts-types séparés.
var motif = new Regex(
    @"seed\s+(\d+)\s*:\s*maison\s+([0-9.]+)->([0-9.]+)\s*\((\d+)s\)\s*\|\s*trl\s+([0-9.]+)->([0-9.]+)\s*\((\d+)s\)");
var series = motif.Matches(SansAccent(txt2)).Cast<Match>().ToList();
var differences = series.Select(m =>
    double.Parse(m.Groups[3].Value, CultureInfo.InvariantCulture)
    - double.Parse(m.Groups[6].Value, CultureInfo.InvariantCulture)).ToList();
var temps_maison = series.Select(m => int.Parse(m.Groups[4].Value)).ToList();
var temps_trl = series.Select(m => int.Parse(m.Groups[7].Value)).ToList();
double moy_d = differences.Average();
double sd_d = EcartType(differences);
// t apparié : moyenne des différences sur l'erreur-type des différences.
double t_apparie_val = moy_d / (sd_d / Math.Sqrt(differences.Count));
Console.WriteLine($"  differences par graine (maison - trl) : [{string.Join(", ", differences.Select(d => Inv(d, "0.000")))}]");
Console.WriteLine($"  moyenne {Inv(moy_d, "+0.0000;-0.0000")} | ecart-type {Inv(sd_d, "0.0000")} | t apparie {Inv(t_apparie_val, "+0.00;-0.00")}");
Console.WriteLine($"  soit, en combinant les deux bras : 2 sigma = {Inv(2 * sd_d, "0.0000")}");
Console.WriteLine($"  temps moyen : maison {temps_maison.Average():0}s | trl {temps_trl.Average():0}s");
Console.WriteLine();

// --- Comparaison 3 : la résolution du jeu de test ---
Console.WriteLine("=== Comparaison 3 : RLPT-04, la resolution du jeu de test ===");
var txt3 = TXT["RLPT-04-DPO-Offline-contre-GRPO-Online-Python.ipynb"];
double base_v = NombreApres(txt3, "BASELINE greedy")!.Value;
double dpo = NombreApres(txt3, "POST-DPO greedy")!.Value;
double grpo = NombreApres(txt3, "POST-GRPO greedy")!.Value;
int n_items = 48;
// Le plus petit pas mesurable : un item sur 48. Un gain plus fin que cela
// n'est pas résolu par l'instrument.
double un_exemple = 1.0 / n_items;
Console.WriteLine($"  baseline {Inv(base_v, "0.000")} | DPO {Inv(dpo, "0.000")} | GRPO {Inv(grpo, "0.000")}  (n = {n_items} items)");
Console.WriteLine($"  1 item = {Inv(un_exemple, "0.0000")} d'exactitude");
Console.WriteLine($"  DPO - baseline : {Inv(dpo - base_v, "+0.0000;-0.0000")} = {Inv((dpo - base_v) / un_exemple, "0.00")} item(s)");
Console.WriteLine($"  DPO - GRPO     : {Inv(dpo - grpo, "+0.0000;-0.0000")} = {Inv((dpo - grpo) / un_exemple, "0.00")} item(s)");
Console.WriteLine($"  erreur-type d'une proportion ~0.75 sur {n_items} items : {Inv(Math.Sqrt(0.75 * 0.25 / n_items), "0.0000")}");
Console.WriteLine();

// --- Comparaison 4 : une métrique qui ne bouge plus ---
Console.WriteLine("=== Comparaison 4 : RLPT-02, une metrique qui ne bouge plus ===");
var txt4 = TXT["RLPT-02-GRPO-Minimal-Python.ipynb"];
var graines = Regex.Matches(
    SansAccent(txt4),
    @"seed\s+\d+\s*:\s*reward moyen\s+([0-9.]+)\s*\|\s*accuracy greedy\s+([0-9.]+)");
var recompenses = graines.Cast<Match>()
    .Select(m => double.Parse(m.Groups[1].Value, CultureInfo.InvariantCulture)).ToList();
var exactitudes = graines.Cast<Match>()
    .Select(m => double.Parse(m.Groups[2].Value, CultureInfo.InvariantCulture)).ToList();
// La récompense bouge d'une graine à l'autre, l'exactitude non : c'est la
// signature d'une optimisation qui n'atteint pas la tâche mesurée.
Console.WriteLine($"  exactitude greedy par graine : [{string.Join(", ", exactitudes.Select(a => Inv(a, "0.00")))}]  -> ecart-type {Inv(EcartType(exactitudes), "0.0000")}");
Console.WriteLine($"  recompense moyenne par graine: [{string.Join(", ", recompenses.Select(r => Inv(r, "0.000")))}]");
Console.WriteLine($"  ecart-type de la recompense  : {Inv(EcartType(recompenses), "0.0000")}");
Console.WriteLine($"  baseline greedy {Inv(NombreApres(txt4, "Baseline greedy")!.Value, "0.00")} | post-RL greedy {Inv(NombreApres(txt4, "Accuracy post-RL (greedy)")!.Value, "0.00")}");


=== Comparaison 1 : RLPT-00d, RewardTrainer TRL contre la reference A.1 ===


  TRL           : 0.6695 +- 0.0180  (n=4 graines)


  reference A.1 : 0.6580 +- 0.0080


  ecart         : +0.0115


  z sur la seule barre A.1 : +1.44   <- le chiffre annonce par le carnet


  z en combinant LES DEUX  : +0.58   <- le chiffre honnete


... (12 outputs left out)

  exactitude greedy par graine : [0.88, 0.88, 0.88, 0.88]  -> ecart-type 0.0000


  recompense moyenne par graine: [0.438, 0.521, 0.667, 0.479]


  ecart-type de la recompense  : 0.0998


  baseline greedy 0.88 | post-RL greedy 0.88


### Lecture des quatre comparaisons

**1. `RLPT-00d` — le gain de 1,2 point est un demi-écart-type.** Le carnet rapporte `+0.012`, soigneusement exprimé en « 1,4 écarts-types A.1 » — mais ce dénominateur ne contient que la barre de la référence, pas celle du bras TRL. Combinées, les deux barres donnent `z = 0,58`. Un gain de 0,58 σ n'est pas un gain : c'est un tirage. Le carnet a raison de dire que les deux protocoles sont comparables ; il ne peut pas dire que l'un surpasse l'autre.

**2. `RLPT-00f` — l'écart apparié est plus petit que son propre bruit.** Les quatre différences par graine changent de signe (deux positives, une négative, une quasi nulle). Le `t` apparié vaut `0,66` : indistinguable de zéro. Le carnet conclut déjà `INCONCLUSIVE`, et le protocole apparié — plus puissant que la comparaison de moyennes qu'il utilise — ne le sauve pas.

**3. `RLPT-04` — 48 items, un item vaut 2,1 points.** C'est la phrase qui manquait. L'écart DPO–GRPO annoncé vaut **exactement un item**. L'écart DPO–baseline en vaut trois. Sur un jeu de 48 questions, l'erreur-type d'une proportion autour de 0,75 est de 6,25 points : toute conclusion qui porte sur moins de deux items est en dessous du plancher de l'instrument. Le carnet ne pouvait pas trancher DPO contre GRPO, et il ne le prétend pas — ce que la sous-série aurait pu dire, c'est **pourquoi**.

**4. `RLPT-02` — la métrique est saturée, donc elle ne mesure plus rien.** Quatre graines, exactitude `0,88` partout, écart-type `0,00`, et identique à la baseline. Ce n'est pas un échec de l'entraînement : c'est un succès de la baseline. Ce qui bouge, c'est la **récompense** — entre 0,44 et 0,67 selon la graine. Autrement dit, GRPO déplace bien quelque chose, mais pas la réponse finale. La conclusion honnête du carnet est donc double : le signal appris vit dans le canal de la récompense, et l'exactitude n'est pas l'instrument qui peut le voir.

Le motif commun est net : **quatre comparaisons, quatre instruments plus grossiers que l'écart mesuré**. Ce n'est pas une coquetterie statistique. C'est la raison pour laquelle la sous-série a construit des plafonds de Bayes explicites — c'est l'objet de la section suivante.

## 3. La part du plafond de Bayes

Quatre carnets établissent un **plafond de Bayes** : l'exactitude qu'atteindrait un juge optimal sur leur monde. C'est la seule échelle qui rende les exactitudes comparables entre carnets — et la seule qui dise combien de points il restait à gagner au moment où l'entraînement a commencé.

On calcule ici la part du plafond atteinte, et la marge résiduelle en points d'exactitude.

In [4]:
// Part du plafond de Bayes atteinte par chaque reward model : accuracy rapportée
// à son plafond, et marge restante en points -- la tâche n'est jamais "gagnée",
// elle est plus ou moins proche d'un plafond qui n'est pas atteignable.
var mesures = new List<(string Nom, double Acc, double Plafond)>();

var txtA = TXT["RLPT-00-Reward-Model-Bradley-Terry-Python.ipynb"];
mesures.Add(("RLPT-00",
    NombreApres(txtA, "vs accuracy")!.Value,
    NombreApres(txtA, "plafond de Bayes empirique E[max(p, 1-p)] (test)")!.Value));

var txtB = TXT["RLPT-00b-Biais-Dataset-Preferences-Python.ipynb"];
mesures.Add(("RLPT-00b",
    NombreApres(txtB, "lenctrl, accuracy apres correction g*len")!.Value,
    NombreApres(txtB, "plafond de Bayes oracle")!.Value));

// RLPT-00d rapporte par graine : on moyenne les deux colonnes soi-même.
var txtD = TXT["RLPT-00d-Reward-Trainer-TRL-Python.ipynb"];
var couples = Regex.Matches(
    SansAccent(txtD),
    @"seed\s+\d+\s*:\s*acc\s+([0-9.]+)\s*\(Bayes\s+([0-9.]+)\)");
var accs = couples.Cast<Match>().Select(m => double.Parse(m.Groups[1].Value, CultureInfo.InvariantCulture)).ToList();
var bayes = couples.Cast<Match>().Select(m => double.Parse(m.Groups[2].Value, CultureInfo.InvariantCulture)).ToList();
mesures.Add(("RLPT-00d", accs.Average(), bayes.Average()));

var txtE = TXT["RLPT-00e-DPO-TRL-Python.ipynb"];
mesures.Add(("RLPT-00e",
    NombreApres(txtE, "accuracy test")!.Value,
    NombreApres(txtE, "plafond d'accuracy (Bayes)")!.Value));

Console.WriteLine($"{"carnet",-10} {"accuracy",9} {"plafond",8} {"part",7} {"marge (pts)",12}");
Console.WriteLine(new string('-', 50));
foreach (var (nom, acc, plafond) in mesures)
{
    double part = acc / plafond;
    Console.WriteLine($"{nom,-10} {Inv(acc, "0.000"),9} {Inv(plafond, "0.000"),8} {Inv(part, "0.0%"),7} {Inv(100 * (plafond - acc), "0.0"),11}");
}
Console.WriteLine();
Console.WriteLine("Lecture : la marge est ce qu'il restait a gagner sur ce monde, pas ce qu'un bras a gagne.");


carnet      accuracy  plafond    part  marge (pts)


--------------------------------------------------


RLPT-00        0.665    0.702   94.7%         3.7


RLPT-00b       0.574    0.699   82.1%        12.5


RLPT-00d       0.670    0.696   96.2%         2.6


RLPT-00e       0.665    0.702   94.7%         3.7


Lecture : la marge est ce qu'il restait a gagner sur ce monde, pas ce qu'un bras a gagne.


### Lecture de la marge

Trois carnets sur quatre terminent entre **93,6 % et 96,2 %** de leur plafond. Il restait donc entre **2,6 et 4,5 points** d'exactitude à gagner sur ces mondes, et les carnets les ont pour l'essentiel pris.

C'est exactement ce qui rend les comparaisons de la section 2 non concluantes, et le lien est arithmétique : un bras qui vaut 96 % d'un plafond ne peut pas en gagner 5 de plus, donc tout écart annoncé dans cette zone est un écart **contre un plafond**, pas contre l'autre bras. `RLPT-00d` annonce 1,2 point dans une marge de 2,6 : le gain occupe près de la moitié de ce qui restait, avec un `z` de 0,58.

`RLPT-00b` est l'exception instructive : **82,1 %** seulement, soit **12,5 points** de marge encore ouverts. Le contrôle de longueur coûte cher — il retire du signal utile en même temps que du biais — et c'est le seul carnet de la sous-série dont le modèle garde une marge substantielle. Il ne s'en cache pas : le carnet présente ce chiffre comme le prix du contrôle, pas comme un échec.

Un plafond n'est pas une excuse. C'est un **dénominateur** : sans lui, « 0,657 d'exactitude » ne veut rien dire, et « +1,2 point » non plus.

## 4. La marge de reward hacking se calcule

`RLPT-00c` est le carnet le plus falsifiable de la sous-série, parce qu'il énonce un **seuil analytique** : le rapport entre le poids de la longueur et celui de la correction vaut `B / THETA`, soit `10 / 6`. Au-delà, l'argmax du proxy devient une réponse incorrecte — le modèle préfère la longueur à l'exactitude, et la politique optimise la mauvaise chose.

Ce seuil a une propriété précieuse : il se recalcule. On peut donc mesurer non pas « y a-t-il eu hacking ? » mais **de combien** le modèle appris a dépassé la frontière.

In [5]:
// Anatomie du reward hacking dans RLPT-00c, relu en quatre instruments :
// seuil analytique, chute de qualité, décomposition par canal, résolution.
var txt = TXT["RLPT-00c-Anatomie-Reward-Hacking-Python.ipynb"];

// 1. Le rapport mesuré dépasse-t-il le seuil analytique B/THETA, et de combien.
double rapport_mesure = NombreApres(txt, "rapport |wl / wc|")!.Value;
var seuilTriplet = NombresApres(txt, "seuil analytique du rapport wl/wc", 3);
double budget = seuilTriplet[0]!.Value;
double theta = seuilTriplet[1]!.Value;
double seuil = seuilTriplet[2]!.Value;
double marge_relative = rapport_mesure / seuil - 1;
Console.WriteLine($"  rapport mesure |wl/wc|   : {Inv(rapport_mesure, "0.0000")}");
Console.WriteLine($"  seuil analytique B/THETA : {budget:0}/{theta:0} = {Inv(seuil, "0.0000")}");
Console.WriteLine($"  marge au-dessus du seuil : {Inv(marge_relative, "+0.0%;-0.0%")}");
Console.WriteLine();

// 2. Ce que chaque bras a fait de la QUALITÉ VRAIE (le proxy n'est pas la tâche).
var qualitePaire = NombresApres(txt, "QUALITE VRAIE", 2);
double qualite_initiale = qualitePaire[0]!.Value;
double qualite_bras1 = qualitePaire[1]!.Value;
double qualite_bras2 = NombreApres(txt, "qualite finale longueur-controle")!.Value;
Console.WriteLine($"  qualite vraie : initiale {Inv(qualite_initiale, "0.0000")} | bras 1 (biaise) {Inv(qualite_bras1, "0.0000")} | bras 2 (longueur-controle) {Inv(qualite_bras2, "0.0000")}");
Console.WriteLine($"  chute bras 1 : {Inv(qualite_bras1 - qualite_initiale, "+0.0000;-0.0000")} | gain bras 2 : {Inv(qualite_bras2 - qualite_initiale, "+0.0000;-0.0000")}");
Console.WriteLine($"  ecart bras 2 - bras 1 : {Inv(qualite_bras2 - qualite_bras1, "+0.0000;-0.0000")}");
Console.WriteLine();

// 3. Décomposition par canal : le gain de proxy doit se fermer en qualité + longueur.
double canal_qualite = NombreApres(txt, "canal qualite")!.Value;
double canal_longueur = NombreApres(txt, "canal longueur")!.Value;
double proxy_somme = NombreApres(txt, "proxy (somme)")!.Value;
double residu = canal_qualite + canal_longueur - proxy_somme;
Console.WriteLine("  decomposition de l'amelioration du proxy :");
Console.WriteLine($"    canal qualite  : {Inv(canal_qualite, "+0.0000;-0.0000")}");
Console.WriteLine($"    canal longueur : {Inv(canal_longueur, "+0.0000;-0.0000")}");
Console.WriteLine($"    somme          : {Inv(canal_qualite + canal_longueur, "+0.0000;-0.0000")}");
Console.WriteLine($"    proxy annonce  : {Inv(proxy_somme, "+0.0000;-0.0000")}");
Console.WriteLine($"    residu         : {Inv(residu, "+0.00e+00;-0.00e+00")}   <- 0 = la decomposition se referme");
Console.WriteLine();

// 4. Résolution : l'encadrement du balayage contre celui de la bissection.
var bissection = Regex.Match(SansAccent(txt), @"\(largeur\s+([0-9.eE+-]+)\)");
double? largeur_bissection = bissection.Success
    ? double.Parse(bissection.Groups[1].Value, CultureInfo.InvariantCulture)
    : (double?)null;
var balayage = Regex.Match(SansAccent(txt), @"de largeur\s+([0-9.]+)");
double? largeur_balayage = balayage.Success
    ? double.Parse(balayage.Groups[1].Value, CultureInfo.InvariantCulture)
    : (double?)null;
Console.WriteLine($"  encadrement du balayage  : largeur {Inv(largeur_balayage!.Value, "0.00")}");
Console.WriteLine($"  encadrement bissection   : largeur {Inv(largeur_bissection!.Value, "0.0e+00")}");
if (largeur_bissection.HasValue && largeur_balayage.HasValue)
{
    Console.WriteLine($"  rapport des deux         : {Inv(largeur_balayage.Value / largeur_bissection.Value, "0.00e+00")} fois plus fin");
}

// Témoin du franchissement : une borne sous le seuil, une borne au-dessus.
var bornes = Regex.Matches(SansAccent(txt), @"wl/wc=([0-9.]+), seuil ([0-9.]+)");
if (bornes.Count >= 2)
{
    double bas = double.Parse(bornes[0].Groups[1].Value, CultureInfo.InvariantCulture);
    double seuil_bas = double.Parse(bornes[0].Groups[2].Value, CultureInfo.InvariantCulture);
    double haut = double.Parse(bornes[1].Groups[1].Value, CultureInfo.InvariantCulture);
    double seuil_haut = double.Parse(bornes[1].Groups[2].Value, CultureInfo.InvariantCulture);
    Console.WriteLine();
    Console.WriteLine("  verification du franchissement de seuil :");
    Console.WriteLine($"    borne basse {Inv(bas, "0.000000")} < seuil {Inv(seuil_bas, "0.000000")} : {(bas < seuil_bas ? "True" : "False")}");
    Console.WriteLine($"    borne haute {Inv(haut, "0.000000")} > seuil {Inv(seuil_haut, "0.000000")} : {(haut > seuil_haut ? "True" : "False")}");
}


  rapport mesure |wl/wc|   : 1.9923


  seuil analytique B/THETA : 10/6 = 1.6667


  marge au-dessus du seuil : +19.5%


  qualite vraie : initiale 0.4545 | bras 1 (biaise) 0.0170 | bras 2 (longueur-controle) 0.9902


  chute bras 1 : -0.4375 | gain bras 2 : +0.5357


  ecart bras 2 - bras 1 : +0.9732


... (8 outputs left out)

  rapport des deux         : 1.67e+07 fois plus fin


  verification du franchissement de seuil :


    borne basse 1.666261 < seuil 1.666667 : True


    borne haute 1.666937 > seuil 1.666667 : True


### Lecture de la marge

Le modèle appris dépasse le seuil analytique de **19,5 %**. Ce n'est pas un ordre de grandeur accidentel : c'est la conséquence directe du protocole d'annotation, où chaque paire est annotée avec un biais `a = 2,0`, et où le carnet montre que le rapport mesuré suit ce biais.

La décomposition par canal est le résultat le plus élégant de la sous-série, et elle **se vérifie** : `-1,7006 + 3,7669 = +2,0663`, exactement le gain de proxy annoncé, résidu nul à la précision d'affichage. Autrement dit, l'amélioration apparente du proxy se décompose intégralement en **une perte de qualité de 1,70** et **un gain de longueur de 3,77** — le second canal payant pour le premier et au-delà. Le bras biaisé ne « progresse » pas : il **convertit** de la qualité en longueur à un taux mesurable.

Le pendant est le bras 2 : contrôler la longueur rend `0,9902` de qualité vraie, soit plus que la politique initiale (`0,4545`) et au niveau de l'oracle (`0,9888`). Le même PPO, sur le même monde, apprend donc exactement ce qu'on lui donne à apprendre — ce qui est la définition opérationnelle d'un problème de spécification, et non d'optimisation.

## 5. Le coût par point de gain

Un écart non concluant n'est pas un écart sans valeur : il peut porter une différence de **coût**. C'est même le seul endroit où un harnais industriel peut gagner sur une boucle maison sans changer les résultats — moins de temps, moins de code à comprendre, moins de surface de bug.

On rapporte donc les deux : secondes par point de gain mesuré, et secondes par point de gain **crédible**.

In [6]:
// Coût par point de gain : le temps de calcul divisé par le gain -- ce que le
// rapport de gain ne dit pas quand l'écart statistique est nul.
var txtF = TXT["RLPT-00f-GRPO-TRL-contre-PPO-Maison-Python.ipynb"];
var maisonTriplet = NombresApres(txtF, "gain maison", 3);
var trlTriplet = NombresApres(txtF, "gain trl", 3);
double gain_maison = maisonTriplet[0]!.Value;
double sd_maison = maisonTriplet[1]!.Value;
double temps_maison_moyen = maisonTriplet[2]!.Value;
double gain_trl = trlTriplet[0]!.Value;
double sd_trl = trlTriplet[1]!.Value;
double temps_trl_moyen = trlTriplet[2]!.Value;
Console.WriteLine("=== RLPT-00f : cout par point de gain ===");
Console.WriteLine($"  maison : gain {Inv(gain_maison, "+0.000;-0.000")} +- {Inv(sd_maison, "0.000")} | {temps_maison_moyen:0}s | {temps_maison_moyen / gain_maison:0} s par point");
Console.WriteLine($"  trl    : gain {Inv(gain_trl, "+0.000;-0.000")} +- {Inv(sd_trl, "0.000")} | {temps_trl_moyen:0}s | {temps_trl_moyen / gain_trl:0} s par point");
Console.WriteLine($"  rapport de cout par point : {Inv((temps_maison_moyen / gain_maison) / (temps_trl_moyen / gain_trl), "0.00")}x en faveur de TRL");
Console.WriteLine("  mais l'ecart de gain entre les bras est nul statistiquement (section 2).");
Console.WriteLine("  -> le gain de TRL est un gain d'ingenierie, pas un gain de resultat.");
Console.WriteLine();

// RLPT-00d : le même temps rapporté à deux dénominateurs différents (boucle
// d'entraînement seule, puis run complet avec évaluation).
var txtD2 = TXT["RLPT-00d-Reward-Trainer-TRL-Python.ipynb"];
double rapport_annonce = NombreApres(txtD2, "rapport de temps trl / maison")!.Value;
double temps_moyen_run = NombreApres(txtD2, "temps moyen par run")!.Value;
var boucle_maison = Regex.Match(SansAccent(txtD2), @"en\s+([0-9.]+)\s*s");
double? boucle_maison_s = boucle_maison.Success
    ? double.Parse(boucle_maison.Groups[1].Value, CultureInfo.InvariantCulture)
    : (double?)null;
Console.WriteLine("=== RLPT-00d : deux denominateurs, deux rapports ===");
Console.WriteLine($"  rapport committe (boucle d'entrainement seule) : {Inv(rapport_annonce, "0.00")}x");
if (boucle_maison_s.HasValue)
{
    Console.WriteLine($"  boucle maison : {Inv(boucle_maison_s.Value, "0.0")}s | run complet moyen : {Inv(temps_moyen_run, "0.0")}s");
    Console.WriteLine($"  rapport en incluant l'evaluation : {Inv(temps_moyen_run / boucle_maison_s.Value, "0.00")}x");
}
Console.WriteLine("  pour un ecart d'exactitude de +0.012, soit 0.58 sigma (section 2).");
Console.WriteLine("  -> 39 secondes de plus par run pour une difference que l'instrument ne resout pas.");


=== RLPT-00f : cout par point de gain ===


  maison : gain +0.635 +- 0.100 | 121s | 191 s par point


  trl    : gain +0.605 +- 0.074 | 80s | 132 s par point


  rapport de cout par point : 1.44x en faveur de TRL


  mais l'ecart de gain entre les bras est nul statistiquement (section 2).


  -> le gain de TRL est un gain d'ingenierie, pas un gain de resultat.


... (2 outputs left out)

  boucle maison : 6.9s | run complet moyen : 45.6s


  rapport en incluant l'evaluation : 6.61x


  pour un ecart d'exactitude de +0.012, soit 0.58 sigma (section 2).


  -> 39 secondes de plus par run pour une difference que l'instrument ne resout pas.


### Lecture du coût

`RLPT-00f` est le cas le plus net de la sous-série, et le plus utile : **TRL coûte 1,44 fois moins cher par point de gain, et son gain est statistiquement identique à celui de la boucle maison**. Les deux affirmations sont vraies en même temps, et c'est précisément ce qu'un ingénieur doit savoir. Le carnet a donc raison de conclure `INCONCLUSIVE` sur le résultat : la conclusion `INCONCLUSIVE` **est** le livrable, puisque ce qui reste mesurable est le coût.

`RLPT-00d` illustre un piège de lecture, sans que le carnet soit en faute. Le rapport committé de `5,95x` porte sur la **boucle d'entraînement** (`41,0 s` contre `6,9 s`) ; rapporté au **run complet** (`45,6 s`), le même rapport vaut `6,61x`. Les deux chiffres sont justes et désignent deux choses différentes. Un capstone sert à cela : nommer le dénominateur, parce qu'un rapport de temps sans son dénominateur est un nombre qui circule sans son sens.

La ligne la plus dure pour l'écosystème est ailleurs : `+0,012` d'exactitude pour 39 secondes de plus par run. Ce n'est pas un mauvais échange — c'est un **échange non mesuré**, et aucun des deux carnets ne prétend le contraire. La valeur de TRL dans cette sous-série est l'ergonomie (`1,8x` moins de lignes à écrire, `2` lignes pour sauver et recharger un checkpoint fidèle au bit près), pas la performance.

## 6. Ce que la sous-série ne démontre pas

Un capstone qui ne dirait que des succès ne serait pas un capstone. Ce que les onze carnets **ne** démontrent pas, mesuré ici :

- **Aucun transfert hors de leur monde.** Les quatre plafonds de Bayes sont *empiriques* : ils sont calculés sur le même générateur que l'entraînement. Un modèle à 98 % de son plafond sur un monde synthétique n'a rien prouvé sur un monde réel — il a prouvé qu'il apprend ce générateur. `RLPT-03` et `RLPT-04` vont plus loin, puisqu'ils travaillent sur un modèle de langage réel, mais leur jeu de test fait 48 items, ce qui plafonne leur résolution à 2,1 points.
- **Aucune comparaison inter-harnais concluante.** Les trois face-à-face TRL/maison (`00d`, `00e`, `00f`) rendent tous un écart non significatif. Ils établissent l'**équivalence** des implémentations, ce qui est un résultat, mais pas une supériorité.
- **Aucune borne sur la graine.** Les quatre graines de `00f` et `00d` sont les mêmes (`0, 1, 7, 42`) ; c'est un bon protocole, mais quatre graines ne donnent qu'un écart-type à trois degrés de liberté, donc des barres d'erreur elles-mêmes incertaines d'un facteur 2 environ. Les conclusions de non-significativité sont robustes ; les tailles d'effet ne le sont pas.
- **Aucune preuve que l'exactitude soit le bon objectif.** `RLPT-02` montre une exactitude figée à `0,88` pendant que la récompense varie de `0,44` à `0,67` : la métrique choisie peut être saturée alors que le modèle apprend encore. C'est le résultat le plus transférable de la sous-série vers un usage réel.

Ce dernier point est aussi la raison d'être de la section 2 : avant de comparer deux bras, il faut savoir **à quelle résolution** on mesure. C'est une question qu'on se pose une fois, et qui vaut pour tous les carnets suivants.

## Exercices

Les trois exercices reconstruisent les instruments de mesure du carnet. Ils portent sur les fonctions, pas sur les nombres : une fois écrites, elles doivent reproduire exactement les valeurs affichées plus haut.

**Exercice 1 — combiner deux barres d'erreur.**

Écrivez `z_combine(m1, s1, m2, s2)` qui rend l'écart réduit entre deux moyennes en combinant **les deux** écarts-types. Vérifiez qu'elle rend `+0,58` sur la comparaison 1, et non `+1,44`.

*Indice* : l'écart des moyennes se divise par la racine de la somme des carrés des deux écarts-types.

In [7]:
// Exercice 1 -- combiner deux barres d'erreur.
// L'écart réduit entre deux moyennes, en combinant LEURS DEUX barres.
double? ZCombine(double m1, double s1, double m2, double s2)
{
    double? result = null;  // TODO etudiant
    return result;
}

Console.WriteLine("Exercice a completer : z_combine");
Console.WriteLine("attendu sur la comparaison 1 : z = +0.58");


Exercice a completer : z_combine


attendu sur la comparaison 1 : z = +0.58


**Exercice 2 — la part du plafond.**

Écrivez `part_du_plafond(accuracy, plafond)` qui rend la fraction du plafond atteinte, et `marge_en_points(accuracy, plafond)` qui rend ce qu'il restait à gagner, en points d'exactitude. Les deux doivent rendre un résultat sur `RLPT-00b` (`0,574` et `0,699`) qui explique pourquoi ce carnet est le seul à garder une marge substantielle.

In [8]:
// Exercice 2 -- la part du plafond de Bayes.
// La fraction du plafond atteinte, entre 0 et 1.
double? PartDuPlafond(double accuracy, double plafond)
{
    double? result = null;  // TODO etudiant
    return result;
}

// Ce qu'il restait à gagner, en points d'exactitude.
double? MargeEnPoints(double accuracy, double plafond)
{
    double? result = null;  // TODO etudiant
    return result;
}

Console.WriteLine("Exercice a completer : part_du_plafond et marge_en_points");
Console.WriteLine("attendu sur RLPT-00b : 82.1% et 12.5 points");


Exercice a completer : part_du_plafond et marge_en_points


attendu sur RLPT-00b : 82.1% et 12.5 points


**Exercice 3 — le t apparié.**

Écrivez `t_apparie(avant, apres)` qui rend la statistique `t` d'un protocole apparié, puis appliquez-la aux quatre graines de `RLPT-00f`. Vous devez retrouver `t = 0,66`.

Question de fond : pourquoi le `t` apparié est-il **plus puissant** que la comparaison des deux moyennes utilisée par le carnet, et pourquoi ne suffit-il pourtant pas à rendre l'écart significatif ici ?

In [9]:
// Exercice 3 -- le t apparié.
// Statistique t d'un protocole apparié (mêmes graines des deux côtés).
double? TApparie(IReadOnlyList<double> avant, IReadOnlyList<double> apres)
{
    double? result = null;  // TODO etudiant
    return result;
}

// Les quatre exactitudes de chaque liste sont recopiées des sorties committées
// de RLPT-00f (graines maison vs graines TRL). Elles sont figées VOLONTAIREMENT,
// pour que l'exercice ait un attendu stable : si RLPT-00f est ré-exécuté, c'est
// ici qu'on compare avant de conclure que l'attendu tient toujours.
var graines_maison = new[] { 0.977, 0.836, 0.839, 0.961 };
var graines_trl = new[] { 0.825, 0.811, 0.905, 0.953 };
Console.WriteLine("Exercice a completer : t_apparie");
Console.WriteLine("attendu sur ces quatre graines : t = +0.66");


Exercice a completer : t_apparie


attendu sur ces quatre graines : t = +0.66


## À retenir

- **La résolution de l'instrument précède la comparaison.** Quatre comparaisons de la sous-série sont non concluantes, et quatre fois la raison est la même : l'écart est plus petit que ce que l'instrument peut départager. Un `z` se calcule avec **les deux** barres d'erreur, pas avec celle de la référence seule — `1,44 σ` devient `0,58 σ` dès qu'on le fait correctement.
- **Un jeu de test de 48 items résout 2,1 points.** Un écart d'un item est du bruit ; deux items commencent à être une mesure. Avant d'interpréter un écart d'exactitude, divisez-le par `1 / n`.
- **Le plafond de Bayes est un dénominateur, pas une excuse.** Trois carnets terminent entre 93,6 % et 96,2 % de leur plafond : il n'y avait presque rien à gagner, et les écarts annoncés dans cette zone sont des écarts contre le plafond. Le seul carnet qui garde une marge (`RLPT-00b`, 82,1 %, soit 12,5 points) la paie au prix du contrôle de longueur.
- **Le reward hacking se mesure.** `RLPT-00c` dépasse son seuil analytique de 19,5 %, et sa décomposition par canal se referme exactement : `-1,70` de qualité pour `+3,77` de longueur, soit `+2,07` de proxy. Le modèle ne progresse pas, il convertit.
- **Une métrique d'exactitude peut être saturée alors que le modèle apprend.** `RLPT-02` le montre : exactitude figée à `0,88` sur quatre graines, récompense qui varie de `0,44` à `0,67`. Choisir la métrique fait partie de l'expérience.
- **Un écart non concluant n'est pas un écart sans valeur.** Il laisse place au coût : `TRL` coûte 1,44 fois moins cher par point, à résultat statistiquement identique. C'est un gain d'ingénierie, et le dire ainsi est plus honnête que de le vendre comme une performance.

## Vérifiez votre compréhension

1. Le carnet `RLPT-00d` annonce `+0,012` d'exactitude et l'exprime en « 1,4 écarts-types A.1 ». Pourquoi ce dénominateur est-il le mauvais, et quel chiffre obtient-on en combinant les deux barres ?
2. Un carnet rapporte `+0,021` d'exactitude sur un jeu de 48 items. Combien d'items cela représente-t-il, et que peut-on en conclure ?
3. Un modèle atteint `0,688` pour un plafond de Bayes de `0,702`. Combien de points restait-il à gagner, et pourquoi un gain annoncé de 2 points dans cette zone est-il suspect ?
4. `RLPT-00c` trouve `|wl/wc| = 1,9923` pour un seuil analytique de `10/6`. Le modèle a-t-il dépassé le seuil, et de combien ?
5. Pourquoi `-1,7006 + 3,7669 = +2,0663` est-il un résultat plus fort qu'une simple liste de trois nombres ?
6. `RLPT-02` obtient `0,88` d'exactitude avec un écart-type inter-graines de `0,00`, et une récompense qui varie de `0,44` à `0,67`. Que mesure réellement l'entraînement, et quelle métrique faudrait-il rapporter ?
7. Pourquoi le `t` apparié de l'exercice 3 est-il plus puissant que la comparaison de deux moyennes indépendantes, et pourquoi ne suffit-il pas ici ?
8. Le rapport de temps de `RLPT-00d` vaut `5,95x` ou `6,61x` selon le dénominateur. Quel est le bon chiffre, et que faut-il toujours écrire à côté d'un rapport de temps ?

## Pour aller plus loin

- **Dans la série RL** : [RL-19-Auto-Jeu-AlphaZero](RL-19-Auto-Jeu-AlphaZero-Python.ipynb) (le carnet précédent — l'auto-jeu quand le réseau guide l'arbre de recherche) et le [README de la série](README.md) pour la place de ce capstone dans l'arc RL.
- **Le plafond de Bayes comme dénominateur** : `RLPT-00` le construit à la main en énumérant les paires et en calculant `E[max(p, 1-p)]` ; c'est la brique qui rend tout le reste interprétable, et elle se réimplémente en une vingtaine de lignes.
- **La décomposition par canal** : `RLPT-00c` sépare le proxy en qualité et longueur, puis vérifie que la somme retombe sur le gain mesuré. C'est le patron à réutiliser dès qu'une récompense est une somme de termes dont un seul est l'objectif.
- **Les protocoles appariés** : `RLPT-00f` fournit quatre graines appariées — la matière première d'un test apparié — mais ne l'exploite pas. L'exercice 3 comble ce manque.
- **La fidélité des checkpoints** : `RLPT-00d` vérifie qu'un modèle rechargé produit exactement les mêmes poids (`0,0e+00` d'écart) et des logits à `2,91e-03`, entièrement expliqués par l'arrondi `bfloat16`. Une différence de logits n'est pas toujours un bug.
- **La fenêtre de disponibilité d'un outil** : `RLPT-00g` documente que `trl.experimental.ppo` a existé de `trl 1.10.0` à `1.12.0` puis a été retiré en `1.13.0`. Un bras négatif documenté vaut mieux qu'un bras absent.
- **Le retour au socle** : l'arc de la sous-série est décrit dans son [README](PostTraining/README.md).